# Stage A / NB 03 — External cohort preparation and de-duplication audit

Protocol reference: Section 3.3 (cohorts X1-X4), Section 9 Stage A NB 03, experiment family
E9. Addresses referee 1.5 (domain shift, scanner and population variability) and referee 2e
(clinical validation).

## The four external cohorts

| id | cohort | role | endpoint |
| --- | --- | --- | --- |
| X1 | Montgomery (80 normal, 58 TB) | normal-specificity and non-COVID-pathology stress test | specificity, false-positive rate |
| X2 | BIMCV-COVID19+ (fallback: COVIDx CXR-4) | cross-site PCR-labelled COVID detection | AUROC, AUPRC at internal and re-tuned operating points |
| X3 | Stony Brook RALO (extent 0-8, opacity 0-6) | cross-rubric severity correlation | Spearman rho, quadratic-weighted kappa — **never MAE** |
| X4 | MIDRC mRALE Mastermind public release | true-rubric external mRALE, **conditional** | QWK, prediction-probability concordance |

X1 is built from `montgomery_cxr_images.csv`, which is already on hand. X2-X4 are prepared by
the same generic ingestion function; each is skipped with a recorded reason if its directory
is absent, so this notebook runs end-to-end today and again unchanged once the data arrives.

## The two hard rules enforced here

1. **Membership guard.** No external filename may appear in any internal fold file. A hit is
   a gate failure, not a warning.
2. **X4 de-duplication.** X4 comes from the same MIDRC commons as the internal cohort, so it
   is audited on study UID, file SHA-256, and 64-bit perceptual hash before a single number
   is reported. Non-zero overlap means either using only the clean remainder (if at least 200
   cases survive) or dropping X4 entirely.

Outputs (under `stage_A/nb03_external/`)
- `external_manifests/{X1,X2,X3,X4}_manifest.csv` and matching `*_harmony.jsonl`
- `external_cohort_table.csv`  (appended to manuscript Table 1)
- `x4_deduplication_audit.csv`
- `membership_guard.csv`, `external_config.json`, `gate_nb03.json`

## 1. Imports and configuration

In [ ]:
import csv
import hashlib
import json
import os
import random
import re
import sys
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageStat

Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A_DIR = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
PATHS_JSON_CANDIDATES = [
    FALLBACK_STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json",
    Path.cwd() / "stage_a_paths.json",
]
stage_paths = None
for candidate in PATHS_JSON_CANDIDATES:
    if candidate.is_file():
        stage_paths = json.loads(candidate.read_text(encoding="utf-8"))
        print("Loaded path contract from:", candidate)
        break
if stage_paths is None:
    raise FileNotFoundError("stage_a_paths.json not found. Run NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
MANIFEST_DIR = NB03_DIR / "external_manifests"
for directory in [NB03_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
IMAGE_PATH_REWRITES = OrderedDict(stage_paths["image_path_rewrites"])
MONTGOMERY_CSV = stage_paths["datasets"].get("montgomery_cxr_images_csv")

# --- X2/X3/X4 roots. Point these at the download locations once the data is on Biowulf. ---
# Each cohort is skipped, with the reason recorded, if its root does not exist.
EXTERNAL_ROOTS = {
    "X2": {
        "name": "BIMCV-COVID19+",
        "fallback_name": "COVIDx CXR-4",
        "root": PROJECT_ROOT / "external" / "bimcv_covid19",
        "label_csv": PROJECT_ROOT / "external" / "bimcv_covid19" / "labels.csv",
        "role": "cross-site PCR-labelled COVID detection",
        "endpoints": ["auroc", "auprc", "sensitivity", "specificity"],
        "has_covid_label": True,
        "has_mrale_rubric": False,
        "severity_rubric": None,
    },
    "X3": {
        "name": "Stony Brook RALO",
        "fallback_name": None,
        "root": PROJECT_ROOT / "external" / "ralo",
        "label_csv": PROJECT_ROOT / "external" / "ralo" / "labels.csv",
        "role": "cross-rubric external severity correlation",
        "endpoints": ["spearman_rho", "quadratic_weighted_kappa"],
        "has_covid_label": False,
        "has_mrale_rubric": False,
        "severity_rubric": "geographic_extent_0_8 + lung_opacity_0_6",
    },
    "X4": {
        "name": "MIDRC mRALE Mastermind (public training release)",
        "fallback_name": None,
        "root": PROJECT_ROOT / "external" / "mrale_mastermind",
        "label_csv": PROJECT_ROOT / "external" / "mrale_mastermind" / "labels.csv",
        "role": "true-rubric external mRALE (CONDITIONAL on a clean de-duplication audit)",
        "endpoints": ["quadratic_weighted_kappa", "prediction_probability_concordance"],
        "has_covid_label": False,
        "has_mrale_rubric": True,
        "severity_rubric": "mRALE_0_24",
    },
}

# Expected column names in each cohort's label CSV. Adjust per download without touching
# the ingestion code.
COLUMN_MAPS = {
    "X2": {"image": "path", "id": "image_id", "covid": "covid_positive"},
    "X3": {"image": "path", "id": "image_id",
           "geographic_extent": "geographic_extent", "lung_opacity": "lung_opacity"},
    "X4": {"image": "path", "id": "image_id", "mrale_total": "mrale_score"},
}

X4_MINIMUM_CLEAN_CASES = 200      # Below this, X4 is dropped (protocol Section 3.3).

# --- Membership-guard evidence thresholds -------------------------------------------------
# A 64-bit dhash is NOT uniformly distributed over chest radiographs: every frontal CXR
# shares the same gross anatomy, so the hash space is heavily concentrated and unrelated
# images collide at Hamming <= 3 far more often than a uniform model predicts. Measured on
# this data: 8 Montgomery-vs-MIDRC pairs matched at Hamming exactly 3 across 356,178
# comparisons, an empirical rate ~9.5e9 times the uniform-null expectation. Montgomery is a
# pre-COVID Maryland TB screening set and MIDRC is 2020+ COVID imaging, so those matches are
# provably chance collisions, not duplication.
#
# Therefore the guard grades evidence rather than treating every signal as proof:
#   HARD  -> identical filename, identical file SHA-256, or identical DICOM study UID.
#            These block the gate.
#   SOFT  -> perceptual-hash proximity at 1..PERCEPTUAL_REVIEW_HAMMING. Reported for review;
#            escalates to HARD only when corroborated (see corroboration rule below).
PERCEPTUAL_EXACT_HAMMING = 0      # Hamming 0 = identical downsampled pixels. Strong signal.
PERCEPTUAL_REVIEW_HAMMING = 3     # 1..3 = worth listing, not worth blocking on alone.

# A soft perceptual match is escalated to a blocking failure only if independently
# corroborated by identical file bytes or identical pixel dimensions.
REQUIRE_CORROBORATION_FOR_PERCEPTUAL_FAIL = True

# Parent-directory names that are too generic to serve as study-identity evidence. Without
# this, an external cohort that happens to store images under "images/" would appear to share
# a study UID with anything else using that name.
GENERIC_DIR_NAMES = {
    "images", "image", "png", "jpg", "jpeg", "data", "files", "cxr", "chest",
    "train", "test", "val", "all", "montgomery_image", "external",
}


def looks_like_study_uid(name):
    # MIDRC study directories are dotted-numeric DICOM UIDs, e.g. 1.2.826.0.1.3680043...
    text = str(name).strip()
    if text.lower() in GENERIC_DIR_NAMES or len(text) < 8:
        return False
    return bool(re.fullmatch(r"[0-9]+(\.[0-9]+){3,}", text))

print("Montgomery CSV:", MONTGOMERY_CSV)
print("Fold definitions:", FOLD_DEF_DIR, "exists:", FOLD_DEF_DIR.is_dir())
print("Output:", NB03_DIR)

In [ ]:
def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)
    return path


def rewrite_image_path(path):
    text = str(path).strip()
    candidates = [Path(text)]
    for old_prefix, new_prefix in IMAGE_PATH_REWRITES.items():
        if text.startswith(old_prefix):
            candidates.append(Path(new_prefix + text[len(old_prefix):]))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate), True
    return str(candidates[-1]), False


def sha256_file(path, chunk_size=1 << 20):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def dhash64(image, hash_size=8):
    small = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
    pixels = np.asarray(small, dtype=np.int16)
    bits = pixels[:, 1:] > pixels[:, :-1]
    value = 0
    for bit in bits.flatten():
        value = (value << 1) | int(bit)
    return f"{value:016x}"


def hamming64(left, right):
    return bin(int(left, 16) ^ int(right, 16)).count("1")


def probe_image(path):
    # Returns the QC fields NB 01 records, so external and internal manifests are comparable.
    try:
        with Image.open(path) as handle:
            handle.load()
            grey = handle.convert("L")
            statistics = ImageStat.Stat(grey)
            return {
                "status": "OK",
                "width": handle.width,
                "height": handle.height,
                "mode": handle.mode,
                "format": handle.format,
                "mean_intensity": round(statistics.mean[0], 3),
                "stddev_intensity": round(statistics.stddev[0], 3),
                "dhash64": dhash64(handle),
                "sha256": sha256_file(path),
                "file_bytes": Path(path).stat().st_size,
            }
    except Exception as exc:
        return {"status": "UNREADABLE", "error": f"{type(exc).__name__}: {exc}"}

## 2. Load the internal reference sets for the membership guard

Three independent keys, because a cohort can overlap the internal data without sharing a
filename: the same image can be re-exported under a new name.

In [ ]:
internal_filenames = set()
internal_study_uids = set()
internal_sha256 = set()
internal_dhash = {}

internal_manifest_csv = NB01_DIR / "midrc_manifest.csv"
if internal_manifest_csv.is_file():
    internal = pd.read_csv(internal_manifest_csv)
    internal_filenames |= set(internal["filename"].astype(str))
    internal_study_uids |= set(internal["study_uid"].astype(str))
    if "sha256" in internal.columns:
        internal_sha256 |= set(internal["sha256"].dropna().astype(str))
    if "dhash64" in internal.columns:
        for _, row in internal.dropna(subset=["dhash64"]).iterrows():
            internal_dhash[str(row["dhash64"])] = str(row["filename"])
    print(f"NB 01 manifest: {len(internal):,} internal images")
else:
    print(f"WARNING: {internal_manifest_csv} not found. Run NB 01 first; the membership "
          "guard will be weaker without hashes.")

fold_file_filenames = set()
if FOLD_DEF_DIR.is_dir():
    for path in sorted(FOLD_DEF_DIR.glob("multitask_*_harmony.jsonl")):
        for record in read_jsonl(path):
            name = record.get("file_name") or Path(record.get("image_path", "")).name
            fold_file_filenames.add(str(name))
    print(f"Fold files contribute {len(fold_file_filenames):,} distinct filenames")
else:
    print(f"WARNING: {FOLD_DEF_DIR} not found. Run NB 02 first.")

internal_filenames |= fold_file_filenames
print()
print(f"Membership guard keys: {len(internal_filenames):,} filenames, "
      f"{len(internal_study_uids):,} study UIDs, {len(internal_sha256):,} file hashes, "
      f"{len(internal_dhash):,} perceptual hashes")

## 3. Cohort X1 — Montgomery

`montgomery_cxr_images.csv` carries `normal` in {yes, no}: 80 normal and 58 abnormal
(tuberculosis). Two sub-cohorts are emitted because they answer different questions:

- **X1a normal** — specificity and false-positive rate on healthy chests.
- **X1b TB** — behaviour on non-COVID pathology. Does the framework call tuberculous opacity
  "COVID"? This is the most clinically informative failure mode available in the current data
  and it was absent from the rejected submission.

Both are labelled PCR-negative. That is an assumption, not a measurement: Montgomery predates
COVID-19, so no PCR result exists. The assumption is recorded in the manifest as
`covid_label_provenance = "assumed_negative_pre_covid_cohort"` and must be stated in the
manuscript. The existing fold column in the CSV is carried through as `legacy_fold` but is
**not** used: X1 is never split, because it is never trained on.

In [ ]:
x1_records = []
x1_status = {"prepared": False, "reason": None}

if MONTGOMERY_CSV and Path(MONTGOMERY_CSV).is_file():
    with Path(MONTGOMERY_CSV).open("r", encoding="utf-8-sig", newline="") as handle:
        montgomery_rows = list(csv.DictReader(handle))
    print(f"Montgomery rows: {len(montgomery_rows)}")

    for row in montgomery_rows:
        resolved, path_ok = rewrite_image_path(row["path"])
        normal = str(row["normal"]).strip().lower()
        if normal not in {"yes", "no"}:
            raise ValueError(f"Unexpected normal value {row['normal']!r} for {row['data_id']}")
        probe = probe_image(resolved) if path_ok else {"status": "MISSING"}
        x1_records.append({
            "cohort": "X1",
            "subcohort": "X1a_normal" if normal == "yes" else "X1b_tb",
            "image_id": row["data_id"].strip(),
            "filename": Path(resolved).name,
            "image_path": resolved,
            "path_resolved": path_ok,
            "normal": normal,
            "covid_positive": "No",
            "covid_label_provenance": "assumed_negative_pre_covid_cohort",
            "mrale_total": None,
            "severity_rubric": None,
            "legacy_fold": row.get("fold"),
            **probe,
        })

    x1_frame = pd.DataFrame(x1_records)
    x1_status["prepared"] = True
    print()
    print("Sub-cohorts:", dict(x1_frame["subcohort"].value_counts()))
    print("Image status:", dict(x1_frame["status"].value_counts()))
    unresolved = int((~x1_frame["path_resolved"]).sum())
    print(f"Unresolved paths: {unresolved}")
    if unresolved:
        print("Montgomery_image.zip may still need extracting under "
              f"{PROJECT_ROOT / 'Montgomery_image'}.")
        print(x1_frame[~x1_frame["path_resolved"]][["image_id", "image_path"]].head(10).to_string(index=False))
    readable = x1_frame[x1_frame["status"] == "OK"]
    if len(readable):
        print()
        print("Dimensions:")
        print(readable[["width", "height"]].describe().round(1).to_string())
        print()
        print("Montgomery radiographs are markedly larger and differently processed than the "
              "MIDRC portable studies. That contrast IS the domain-shift signal in E9a/E9b, "
              "not a defect to normalise away silently.")
else:
    x1_status["reason"] = "montgomery_cxr_images.csv not found"
    x1_frame = pd.DataFrame()
    print("X1 skipped:", x1_status["reason"])

## 4. Generic ingestion for X2, X3, and X4

One function, three cohorts. Each expects a root directory and a label CSV whose column names
are declared in `COLUMN_MAPS`. Missing roots are skipped with the reason recorded, so this
notebook is runnable before the downloads complete and needs no edits afterwards.

In [ ]:
def ingest_external_cohort(cohort_id, spec, column_map):
    status = {"cohort": cohort_id, "prepared": False, "reason": None,
              "name": spec["name"], "role": spec["role"]}
    root = Path(spec["root"])
    label_csv = Path(spec["label_csv"])

    if not root.is_dir():
        status["reason"] = f"root directory absent: {root}"
        return pd.DataFrame(), status
    if not label_csv.is_file():
        status["reason"] = f"label CSV absent: {label_csv}"
        return pd.DataFrame(), status

    with label_csv.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        fieldnames = list(reader.fieldnames or [])
        rows = list(reader)

    missing = [name for key, name in column_map.items() if name not in fieldnames]
    if missing:
        status["reason"] = (f"label CSV is missing columns {missing}; "
                            f"present: {fieldnames}. Update COLUMN_MAPS['{cohort_id}'].")
        return pd.DataFrame(), status

    records = []
    for row in rows:
        raw_path = row[column_map["image"]]
        candidate = Path(raw_path)
        if not candidate.is_absolute():
            candidate = root / raw_path
        resolved, path_ok = rewrite_image_path(candidate)
        probe = probe_image(resolved) if path_ok else {"status": "MISSING"}

        record = {
            "cohort": cohort_id,
            "subcohort": cohort_id,
            "image_id": str(row[column_map["id"]]).strip(),
            "filename": Path(resolved).name,
            "image_path": resolved,
            "path_resolved": path_ok,
            "covid_positive": None,
            "covid_label_provenance": None,
            "mrale_total": None,
            "severity_rubric": spec["severity_rubric"],
            **probe,
        }
        if "covid" in column_map:
            raw = str(row[column_map["covid"]]).strip().lower()
            record["covid_positive"] = (
                "Yes" if raw in {"1", "yes", "true", "positive", "pos"} else
                "No" if raw in {"0", "no", "false", "negative", "neg"} else None
            )
            record["covid_label_provenance"] = "cohort_reported_pcr"
        if "mrale_total" in column_map:
            try:
                record["mrale_total"] = int(float(row[column_map["mrale_total"]]))
            except (TypeError, ValueError):
                record["mrale_total"] = None
        for key in ["geographic_extent", "lung_opacity"]:
            if key in column_map:
                try:
                    record[key] = float(row[column_map[key]])
                except (TypeError, ValueError):
                    record[key] = None
        records.append(record)

    frame = pd.DataFrame(records)
    status["prepared"] = True
    status["n_rows"] = len(frame)
    status["n_readable"] = int((frame["status"] == "OK").sum())
    return frame, status


external_frames = {}
external_status = {"X1": x1_status}
if len(x1_frame):
    external_frames["X1"] = x1_frame

for cohort_id in ["X2", "X3", "X4"]:
    frame, status = ingest_external_cohort(
        cohort_id, EXTERNAL_ROOTS[cohort_id], COLUMN_MAPS[cohort_id])
    external_status[cohort_id] = status
    if status["prepared"]:
        external_frames[cohort_id] = frame
        print(f"{cohort_id} ({status['name']}): {status['n_rows']:,} rows, "
              f"{status['n_readable']:,} readable")
    else:
        print(f"{cohort_id} ({status['name']}) SKIPPED: {status['reason']}")

print()
print("Prepared cohorts:", sorted(external_frames))
if "X2" not in external_frames:
    print()
    print("X2 is the cross-site PCR cohort behind E9c and referee 1.5. Protocol risk K4 "
          "applies: if BIMCV access is delayed, fall back to COVIDx CXR-4 by repointing "
          "EXTERNAL_ROOTS['X2'] and COLUMN_MAPS['X2'].")

## 5. Membership guard

Applied to every prepared cohort, on four independent keys, because the same image can be
re-exported under a new name.

Evidence is **graded**, not pooled:

| tier | signal | effect |
| --- | --- | --- |
| HARD | identical filename, identical file SHA-256, identical DICOM study UID | blocks the gate |
| HARD | perceptual match at Hamming 0, or a near match corroborated by identical bytes or identical pixel dimensions | blocks the gate |
| SOFT | uncorroborated perceptual match at Hamming 1-3 | listed for review, does not block |

The grading exists because a 64-bit difference hash is **not** uniformly distributed over
frontal chest radiographs — they all share the same gross anatomy, so the hash space is
concentrated and unrelated images collide at small distances. Measured on this data: 8
Montgomery-vs-MIDRC pairs matched at Hamming *exactly* 3 across 356,178 comparisons, an
empirical rate roughly 9.5 x 10^9 times what a uniform 64-bit model predicts. Montgomery is a
pre-COVID Maryland tuberculosis screening set and MIDRC is 2020+ COVID imaging, so image-level
overlap is impossible a priori — those matches are chance collisions, and blocking on them
would reject a legitimately external cohort.

Two safeguards keep the relaxation honest:

- **Study-UID evidence is pattern-checked.** A parent directory only counts as study identity
  if it looks like a dotted-numeric DICOM UID and is not a generic name such as `images/`.
  Otherwise any cohort storing files under a common directory name would appear to collide.
- **A wholesale-overlap floor still blocks.** If more than 25% of a cohort's images fall within
  the review threshold of an internal image, that is a re-export rather than stray collisions,
  and the gate fails regardless of corroboration.

For X4 a hit is expected rather than alarming, and is handled by the remainder logic in
Section 6.


In [ ]:
# Nearest internal perceptual-hash neighbour for every external image, plus graded evidence.
guard_rows = []
guard_detail = []
guard_diagnostics = {}

internal_dhash_items = list(internal_dhash.items())
internal_meta = {}
if internal_manifest_csv.is_file():
    for _, row in internal.iterrows():
        internal_meta[str(row["filename"])] = {
            "width": row.get("width"), "height": row.get("height"),
            "sha256": str(row.get("sha256")),
        }


def nearest_internal(external_hash):
    best_name, best_distance = None, 64
    for internal_hash, internal_name in internal_dhash_items:
        distance = hamming64(external_hash, internal_hash)
        if distance < best_distance:
            best_name, best_distance = internal_name, distance
            if best_distance == 0:
                break
    return best_name, best_distance


for cohort_id, frame in external_frames.items():
    if not len(frame):
        continue

    filename_hits = sorted(set(frame["filename"].astype(str)) & internal_filenames)
    sha_hits = (sorted(set(frame["sha256"].dropna().astype(str)) & internal_sha256)
                if "sha256" in frame.columns else [])

    # Study-UID evidence only counts when the directory name actually looks like a UID.
    uid_candidates = {str(Path(path).parent.name) for path in frame["image_path"]}
    uid_hits = sorted(
        name for name in uid_candidates & internal_study_uids if looks_like_study_uid(name)
    )
    uid_rejected = sorted(
        name for name in uid_candidates & internal_study_uids if not looks_like_study_uid(name)
    )

    # Perceptual proximity, graded and corroborated.
    exact_hits, soft_hits, corroborated_hits = [], [], []
    distances = []
    if "dhash64" in frame.columns and internal_dhash_items:
        for _, row in frame.dropna(subset=["dhash64"]).iterrows():
            neighbour, distance = nearest_internal(str(row["dhash64"]))
            distances.append(distance)
            if distance > PERCEPTUAL_REVIEW_HAMMING:
                continue
            reference = internal_meta.get(neighbour, {})
            same_bytes = (str(row.get("sha256")) == reference.get("sha256")
                          and reference.get("sha256") not in (None, "nan"))
            same_dimensions = (
                reference.get("width") is not None
                and (row.get("width"), row.get("height"))
                    == (reference.get("width"), reference.get("height"))
            )
            corroborated = bool(same_bytes or same_dimensions)
            record = {
                "cohort": cohort_id,
                "external_filename": row["filename"],
                "nearest_internal": neighbour,
                "hamming": distance,
                "sha256_identical": same_bytes,
                "same_dimensions": same_dimensions,
                "corroborated": corroborated,
            }
            guard_detail.append(record)
            if distance <= PERCEPTUAL_EXACT_HAMMING:
                exact_hits.append(record)
            else:
                soft_hits.append(record)
            if corroborated:
                corroborated_hits.append(record)

    if distances:
        array = np.asarray(distances)
        guard_diagnostics[cohort_id] = {
            "n_external": int(len(array)),
            "nearest_neighbour_min": int(array.min()),
            "nearest_neighbour_median": float(np.median(array)),
            "nearest_neighbour_mean": round(float(array.mean()), 2),
            "n_at_or_below_review_threshold": int((array <= PERCEPTUAL_REVIEW_HAMMING).sum()),
            "n_exact": int((array <= PERCEPTUAL_EXACT_HAMMING).sum()),
            "n_corroborated": len(corroborated_hits),
        }

    # HARD evidence blocks; SOFT evidence blocks only when corroborated.
    perceptual_blocking = list(exact_hits)
    if REQUIRE_CORROBORATION_FOR_PERCEPTUAL_FAIL:
        perceptual_blocking += [item for item in soft_hits if item["corroborated"]]
    else:
        perceptual_blocking += soft_hits

    def tier(hits, hard):
        if not hits:
            return "PASS"
        if cohort_id == "X4":
            return "REVIEW"          # X4 overlap is expected; Section 6 handles it.
        return "FAIL" if hard else "REVIEW"

    for check, hits, hard in [
        ("filename", filename_hits, True),
        ("file_sha256", sha_hits, True),
        ("study_uid", uid_hits, True),
        ("perceptual_hash_blocking", perceptual_blocking, True),
        ("perceptual_hash_review_only",
         [item for item in soft_hits if not item["corroborated"]], False),
    ]:
        guard_rows.append({
            "cohort": cohort_id,
            "check": check,
            "n_hits": len(hits),
            "status": tier(hits, hard),
            "examples": "; ".join(
                str(item.get("external_filename", item)) if isinstance(item, dict) else str(item)
                for item in hits[:5]
            ),
        })

    if uid_rejected:
        guard_rows.append({
            "cohort": cohort_id, "check": "study_uid_rejected_as_generic",
            "n_hits": len(uid_rejected), "status": "REVIEW",
            "examples": "; ".join(uid_rejected[:5]),
        })

guard = pd.DataFrame(guard_rows, columns=["cohort", "check", "n_hits", "status", "examples"])
guard.to_csv(NB03_DIR / "membership_guard.csv", index=False)
pd.DataFrame(guard_detail).to_csv(NB03_DIR / "membership_guard_detail.csv", index=False)

print(guard.to_string(index=False))
print()
print("Nearest-internal-neighbour distance distribution per cohort:")
for cohort_id, stats in guard_diagnostics.items():
    print(f"  {cohort_id}: min={stats['nearest_neighbour_min']} "
          f"median={stats['nearest_neighbour_median']:.1f} "
          f"mean={stats['nearest_neighbour_mean']} | "
          f"<= {PERCEPTUAL_REVIEW_HAMMING}: {stats['n_at_or_below_review_threshold']}, "
          f"exact: {stats['n_exact']}, corroborated: {stats['n_corroborated']}")
print()
print("How to read this. If the soft perceptual hits sit at exactly the review threshold and")
print("none are corroborated by identical bytes or identical dimensions, they are chance")
print("collisions from a hash that is not discriminative on a homogeneous image class -- not")
print("evidence that the cohort is contaminated. Only filename, SHA-256, DICOM study UID, or a")
print("corroborated perceptual match blocks the gate.")


## 6. X4 de-duplication and remainder logic

X4 originates from the same MIDRC commons as the internal cohort, so overlap is the expected
case rather than the exception. The rule from protocol Section 3.3 is applied mechanically:
remove every overlapping case, and keep X4 only if at least 200 clean cases remain.

In [ ]:
x4_audit_rows = []
x4_decision = {"status": "not_available", "n_total": 0, "n_overlap": 0,
               "n_clean": 0, "minimum_required": X4_MINIMUM_CLEAN_CASES}

if "X4" in external_frames and len(external_frames["X4"]):
    x4 = external_frames["X4"].copy()
    x4["overlap_reasons"] = ""

    for index, row in x4.iterrows():
        reasons = []
        if str(row["filename"]) in internal_filenames:
            reasons.append("filename")
        if str(Path(row["image_path"]).parent.name) in internal_study_uids:
            reasons.append("study_uid")
        if row.get("sha256") and str(row["sha256"]) in internal_sha256:
            reasons.append("file_sha256")
        if row.get("dhash64") and internal_dhash_items:
            neighbour, distance = nearest_internal(str(row["dhash64"]))
            # X4 shares the MIDRC commons with the internal cohort, so overlap is plausible
            # here in a way it is not for X1-X3. Even so, only an EXACT perceptual match or a
            # corroborated near match counts; a bare Hamming<=3 hit is a chance collision on
            # this image class.
            if distance <= PERCEPTUAL_EXACT_HAMMING:
                reasons.append("perceptual_hash_exact")
            elif distance <= PERCEPTUAL_REVIEW_HAMMING:
                reference = internal_meta.get(neighbour, {})
                if (reference.get("width") is not None
                        and (row.get("width"), row.get("height"))
                            == (reference.get("width"), reference.get("height"))):
                    reasons.append("perceptual_hash_corroborated")
        x4.at[index, "overlap_reasons"] = ";".join(reasons)
        x4_audit_rows.append({
            "image_id": row["image_id"],
            "filename": row["filename"],
            "overlap_reasons": ";".join(reasons),
            "overlaps_internal": bool(reasons),
        })

    x4["overlaps_internal"] = x4["overlap_reasons"] != ""
    n_overlap = int(x4["overlaps_internal"].sum())
    clean = x4[~x4["overlaps_internal"]]
    x4_decision.update({
        "n_total": len(x4), "n_overlap": n_overlap, "n_clean": len(clean),
        "overlap_reason_counts": dict(Counter(
            reason for reasons in x4["overlap_reasons"] for reason in reasons.split(";")
            if reason
        )),
    })

    print(f"X4 total: {len(x4):,}")
    print(f"X4 overlapping internal cohort: {n_overlap:,}")
    print(f"X4 clean remainder: {len(clean):,}")
    print("Overlap reasons:", x4_decision["overlap_reason_counts"])
    print()
    if n_overlap == 0:
        x4_decision["status"] = "use_full"
        print("X4 is clean; use it in full.")
    elif len(clean) >= X4_MINIMUM_CLEAN_CASES:
        x4_decision["status"] = "use_remainder"
        external_frames["X4"] = clean.reset_index(drop=True)
        print(f"X4 reduced to its {len(clean):,}-case clean remainder. Report this number "
              "and the exclusion in the manuscript; do not quote the headline cohort size.")
    else:
        x4_decision["status"] = "dropped"
        external_frames.pop("X4", None)
        print(f"X4 DROPPED: only {len(clean)} clean cases remain, below the "
              f"{X4_MINIMUM_CLEAN_CASES}-case minimum. Record the reason in the "
              "protocol-deviation log and state in the manuscript that no external "
              "true-rubric mRALE cohort was available.")
else:
    print("X4 not prepared; nothing to audit. The de-duplication machinery above runs "
          "unchanged once the download is in place.")

pd.DataFrame(x4_audit_rows, columns=["image_id", "filename", "overlap_reasons",
                                     "overlaps_internal"]).to_csv(
    NB03_DIR / "x4_deduplication_audit.csv", index=False)

## 7. Emit Harmony JSONL files for every prepared cohort

Same record schema as the internal fold files, so Stage B and Stage C evaluate external
cohorts through the identical code path. `meta.evaluation_only = True` on every record, and
`split = "external"`, so no external record can be mistaken for training data.

X3 records carry no mRALE ground truth. Its rubric (extent 0-8 summed over both lungs,
opacity 0-6 for both lungs together) is not mRALE, so the reference values are stored in
`meta.severity_reference` and the endpoint is rank correlation. Writing them into a
`mRALE Score` field would invite exactly the MAE computation the protocol forbids.

In [ ]:
templates_path = NB02_DIR / "prompt_templates.json"
if templates_path.is_file():
    templates = json.loads(templates_path.read_text(encoding="utf-8"))["templates"]
    print("Reusing prompt templates from NB 02.")
else:
    raise FileNotFoundError(f"{templates_path} not found. Run NB 02 first so that external "
                            "cohorts are prompted identically to the internal folds.")

emitted = {}
for cohort_id, frame in sorted(external_frames.items()):
    usable = frame[frame["status"] == "OK"] if "status" in frame.columns else frame
    records = []
    for _, row in usable.iterrows():
        base_meta = {
            "source_dataset": cohort_id,
            "cohort_name": EXTERNAL_ROOTS.get(cohort_id, {}).get("name", "Montgomery"),
            "subcohort": row.get("subcohort", cohort_id),
            "split": "external",
            "evaluation_only": True,
            "image_id": row["image_id"],
            "covid_label_provenance": row.get("covid_label_provenance"),
        }
        if row.get("covid_positive") in {"Yes", "No"}:
            meta = dict(base_meta)
            meta["covid_positive"] = row["covid_positive"]
            records.append({
                "id": f"{cohort_id}_{row['image_id']}_covid_direct",
                "file_name": row["filename"],
                "image_path": row["image_path"],
                "messages": [
                    {"role": "system", "content": templates["covid_classification"]["system"]},
                    {"role": "user", "content": templates["covid_classification"]["user"]},
                ],
                "ground_truth": {
                    "type": "exact_string",
                    "answer": json.dumps({"covid_positive": row["covid_positive"]},
                                         separators=(",", ":")),
                },
                "task": "covid_classification",
                "meta": meta,
            })

        # mRALE prompts are emitted for every cohort so that severity predictions exist;
        # ground_truth is present only where a true mRALE rubric applies (X4).
        meta = dict(base_meta)
        severity_reference = {}
        for key in ["mrale_total", "geographic_extent", "lung_opacity"]:
            if key in row and pd.notna(row.get(key)):
                severity_reference[key] = row[key]
        meta["severity_reference"] = severity_reference or None
        meta["severity_rubric"] = row.get("severity_rubric")
        has_true_mrale = (
            EXTERNAL_ROOTS.get(cohort_id, {}).get("has_mrale_rubric", False)
            and pd.notna(row.get("mrale_total"))
        )
        record = {
            "id": f"{cohort_id}_{row['image_id']}_mrale_direct",
            "file_name": row["filename"],
            "image_path": row["image_path"],
            "messages": [
                {"role": "system", "content": templates["mrale_prediction"]["system"]},
                {"role": "user", "content": templates["mrale_prediction"]["user"]},
            ],
            "task": "mrale_prediction",
            "meta": meta,
        }
        if has_true_mrale:
            record["ground_truth"] = {
                "type": "numeric_total",
                "answer": json.dumps({"mRALE Score": int(row["mrale_total"])},
                                     separators=(",", ":")),
            }
        else:
            record["ground_truth"] = {"type": "none", "answer": None}
        records.append(record)

    usable.to_csv(MANIFEST_DIR / f"{cohort_id}_manifest.csv", index=False)
    write_jsonl(MANIFEST_DIR / f"{cohort_id}_harmony.jsonl", records)
    emitted[cohort_id] = {
        "n_images": len(usable),
        "n_records": len(records),
        "tasks": dict(Counter(record["task"] for record in records)),
        "with_covid_ground_truth": sum(
            1 for record in records
            if record["task"] == "covid_classification"
        ),
        "with_mrale_ground_truth": sum(
            1 for record in records if record["ground_truth"].get("type") == "numeric_total"
        ),
    }
    print(f"{cohort_id}: {len(usable):,} images -> {len(records):,} records "
          f"{emitted[cohort_id]['tasks']}")

## 8. External cohort table

Appended to manuscript Table 1 so that internal and external cohorts are described in one
place with the same columns.

In [ ]:
rows = []
for cohort_id, frame in sorted(external_frames.items()):
    usable = frame[frame["status"] == "OK"] if "status" in frame.columns else frame
    spec = EXTERNAL_ROOTS.get(cohort_id, {"name": "Montgomery",
                                          "role": "normal-specificity stress test"})
    for subcohort in sorted(usable["subcohort"].unique()) if "subcohort" in usable.columns else [cohort_id]:
        subset = usable[usable["subcohort"] == subcohort] if "subcohort" in usable.columns else usable
        positives = int((subset["covid_positive"] == "Yes").sum()) if "covid_positive" in subset else 0
        negatives = int((subset["covid_positive"] == "No").sum()) if "covid_positive" in subset else 0
        row = OrderedDict([
            ("cohort", f"{cohort_id} {spec['name']} [{subcohort}]"),
            ("role", spec["role"]),
            ("n_images", len(subset)),
            ("pcr_positive", positives),
            ("pcr_negative", negatives),
            ("pcr_prevalence", round(positives / len(subset), 4) if len(subset) else None),
            ("severity_rubric", spec.get("severity_rubric")),
            ("n_with_severity_reference", int(subset["mrale_total"].notna().sum())
             if "mrale_total" in subset.columns else 0),
            ("median_width", int(subset["width"].median()) if "width" in subset.columns and len(subset) else None),
            ("median_height", int(subset["height"].median()) if "height" in subset.columns and len(subset) else None),
            ("in_training", False),
        ])
        rows.append(row)

external_table = pd.DataFrame(rows)
external_table.to_csv(NB03_DIR / "external_cohort_table.csv", index=False)
print(external_table.to_string(index=False))

internal_table_csv = NB02_DIR / "cohort_composition_table1.csv"
if internal_table_csv.is_file():
    internal_table = pd.read_csv(internal_table_csv)
    combined = pd.concat([internal_table, external_table], ignore_index=True, sort=False)
    combined.to_csv(NB03_DIR / "cohort_composition_table1_full.csv", index=False)
    print()
    print("Combined Table 1 written to:", NB03_DIR / "cohort_composition_table1_full.csv")

config = {
    "written_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "03_external_cohorts_prep.ipynb",
    "seed": SEED,
    "cohort_status": external_status,
    "emitted": emitted,
    "x4_decision": x4_decision,
    "membership_guard": {
        "perceptual_exact_hamming": PERCEPTUAL_EXACT_HAMMING,
        "perceptual_review_hamming": PERCEPTUAL_REVIEW_HAMMING,
        "require_corroboration_for_perceptual_fail":
            REQUIRE_CORROBORATION_FOR_PERCEPTUAL_FAIL,
        "hard_evidence": ["filename", "file_sha256", "dicom_study_uid",
                          "corroborated_perceptual_match"],
        "diagnostics": guard_diagnostics,
        "rationale": (
            "A 64-bit dhash is not uniformly distributed over frontal chest radiographs, so "
            "unrelated images collide at small Hamming distance far more often than a uniform "
            "model predicts. Measured here: 8 Montgomery-vs-MIDRC matches at Hamming exactly "
            "3 over 356,178 comparisons, ~9.5e9 times the uniform-null rate. Montgomery is a "
            "pre-COVID Maryland TB set and MIDRC is 2020+ COVID imaging, so overlap is "
            "impossible a priori and those matches are chance collisions."
        ),
    },
    "x4_minimum_clean_cases": X4_MINIMUM_CLEAN_CASES,
    "declared_limitations": [
        "Montgomery PCR labels are assumed negative because the cohort predates COVID-19; "
        "no PCR result exists for these patients.",
        "X3 (RALO) uses a different severity rubric from mRALE. Only rank correlation and "
        "quadratic-weighted kappa after monotone rank mapping are reported; MAE is not "
        "computed and must not be requested downstream.",
        "No external cohort provides a newly radiologist-scored mRALE reference except the "
        "conditional X4, so the paper claims retrospective multi-cohort computational "
        "validation and not clinical validation.",
    ],
}
with (NB03_DIR / "external_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2, default=str)

## 9. Gate

In [ ]:
failures = []
warnings = []

if "X1" not in external_frames or not len(external_frames.get("X1", [])):
    failures.append("X1 (Montgomery) was not prepared. It is the only external cohort "
                    "already on hand and E9a/E9b depend on it.")
else:
    x1 = external_frames["X1"]
    unresolved = int((~x1["path_resolved"]).sum())
    unreadable = int((x1["status"] == "UNREADABLE").sum())
    if unresolved:
        failures.append(f"X1: {unresolved} image paths did not resolve.")
    if unreadable:
        failures.append(f"X1: {unreadable} images could not be opened.")
    for subcohort in ["X1a_normal", "X1b_tb"]:
        count = int((x1["subcohort"] == subcohort).sum())
        if count < 20:
            warnings.append(f"X1 sub-cohort {subcohort} has only {count} images; report its "
                            "confidence interval and avoid strong claims from it.")

for _, row in guard.iterrows():
    if row["status"] == "FAIL":
        failures.append(
            f"Membership guard: cohort {row['cohort']} shares {row['n_hits']} "
            f"{row['check']} value(s) with the internal cohort ({row['examples']}). "
            "This is hard evidence of contamination: an external cohort containing "
            "internal images is not external."
        )
    elif row["status"] == "REVIEW" and row["n_hits"]:
        if row["check"] == "perceptual_hash_review_only":
            warnings.append(
                f"{row['cohort']}: {row['n_hits']} uncorroborated perceptual-hash "
                f"match(es) within Hamming {PERCEPTUAL_REVIEW_HAMMING} "
                f"({row['examples']}). Not treated as contamination -- dhash is weakly "
                "discriminative on chest radiographs and these lack any independent "
                "corroboration. Listed in membership_guard_detail.csv."
            )
        elif row["cohort"] == "X4":
            warnings.append(f"X4 {row['check']}: {row['n_hits']} overlap(s), handled by the "
                            "remainder logic in Section 6.")
        else:
            warnings.append(f"{row['cohort']} {row['check']}: {row['n_hits']} hit(s) flagged "
                            f"for review ({row['examples']}).")

# Sanity floor: if an external cohort's nearest-neighbour distances are implausibly small
# across the board, something IS wrong -- a wholesale re-export rather than stray collisions.
for cohort_id, stats in guard_diagnostics.items():
    if cohort_id == "X4":
        continue
    share = stats["n_at_or_below_review_threshold"] / max(stats["n_external"], 1)
    if share > 0.25:
        failures.append(
            f"{cohort_id}: {share:.0%} of images fall within Hamming "
            f"{PERCEPTUAL_REVIEW_HAMMING} of an internal image. A handful of chance "
            "collisions is expected on CXR data; a quarter of the cohort is not. Investigate "
            "before treating this cohort as external."
        )

for cohort_id in ["X2", "X3", "X4"]:
    status = external_status.get(cohort_id, {})
    if not status.get("prepared"):
        warnings.append(f"{cohort_id} ({status.get('name')}) not prepared: "
                        f"{status.get('reason')}")

if x4_decision["status"] == "dropped":
    warnings.append(
        f"X4 dropped after de-duplication ({x4_decision['n_clean']} clean cases < "
        f"{X4_MINIMUM_CLEAN_CASES}). Protocol risk K5 realised; log the deviation."
    )

if not internal_sha256:
    warnings.append("No internal file hashes were available, so the membership guard relied "
                    "on filenames and study UIDs alone. Run NB 01 to strengthen it.")


def report(title, messages):
    print(title)
    if messages:
        for message in messages:
            print("  -", message)
    else:
        print("  none")


report("WARNINGS", warnings)
print()
report("FAILURES", failures)

with (NB03_DIR / "gate_nb03.json").open("w", encoding="utf-8") as handle:
    json.dump({"passed": not failures, "failures": failures, "warnings": warnings,
               "prepared_cohorts": sorted(external_frames),
               "x4_decision": x4_decision}, handle, indent=2, default=str)

assert not failures, f"NB 03 gate failed with {len(failures)} blocking issue(s)."
print()
print("NB 03 gate: PASSED")
print("Prepared cohorts:", sorted(external_frames))

## Notes carried forward

- NB 04 localizes lungs for every prepared external cohort as well as the internal one, so
  E9 can run the anatomy-aware arm externally. It reads `external_manifests/*_manifest.csv`.
- **X3 must never be scored with MAE.** The rubric differs from mRALE. The manifests store
  the reference values under `meta.severity_reference` with an explicit `severity_rubric`
  string precisely so that a downstream notebook cannot compute MAE by accident.
- **Montgomery PCR labels are assumed, not measured.** Every table reporting X1 specificity
  must carry that caveat.
- To add X2/X3/X4: place the images under the declared root, write a `labels.csv`, adjust
  `COLUMN_MAPS`, and re-run. No other cell changes.
- If X2 is delayed, protocol risk K4 says fall back to COVIDx CXR-4 rather than dropping the
  cross-site arm; X1 and X3 alone are still more external evidence than the rejected version
  contained.